# Base-model baseline for the reduced-schema pilot
FP32 Gemma 270M; 100 development cases. Greedy decoding with generated-only repetition penalty 1.08, fixed before evaluation.

## Install the established clinical dependencies

In [1]:
%pip install -q "transformers==4.57.6" "datasets==4.8.5" "peft==0.18.1" "accelerate==1.15.0" pandas huggingface_hub nbclient nbformat


## Verify data and freeze the comparison contract

In [2]:
import os, sys
from pathlib import Path
os.chdir("/content/clinical_v4_5000")
sys.path.insert(0, str(Path.cwd()))
import os, json, hashlib, random, re, time, shutil, importlib.metadata
from pathlib import Path
from getpass import getpass
from collections import Counter
import pandas as pd
import torch
from datasets import load_from_disk
from huggingface_hub import get_token
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    LogitsProcessor,
    LogitsProcessorList,
)

def digest(value):
    return hashlib.sha256(json.dumps(value, ensure_ascii=False, sort_keys=True).encode()).hexdigest()

def file_sha(path):
    sha = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            sha.update(block)
    return sha.hexdigest()

def read_token():
    token = os.environ.get("HF_TOKEN") or get_token()
    if not token:
        raise RuntimeError("Missing HF_TOKEN; configure credentials before execution.")
    if not token:
        raise RuntimeError("A Hugging Face token with accepted Gemma access is required.")
    os.environ["HF_TOKEN"] = token
    return token

def runtime_path(path, sentinel):
    path = Path(path).expanduser()
    candidates = [path]
    if not path.is_absolute():
        candidates += [parent / path for parent in Path.cwd().parents]
        candidates.append(Path("/content") / path)
    for candidate in candidates:
        if (candidate / sentinel).is_file():
            return candidate.resolve()
    raise FileNotFoundError(f"Kernel directory: {Path.cwd()}; missing {sentinel} under {path}.")

class GeneratedOnlyRepetitionPenalty(LogitsProcessor):
    """Applies standard repetition penalty only to generated post-prompt tokens."""
    def __init__(self, penalty: float, prompt_length: int):
        if penalty <= 0:
            raise ValueError("penalty must be > 0")
        self.penalty = float(penalty)
        self.prompt_length = int(prompt_length)

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor) -> torch.FloatTensor:
        if self.penalty == 1.0 or input_ids.shape[1] <= self.prompt_length:
            return scores
        gen_ids = input_ids[:, self.prompt_length :]
        if gen_ids.shape[0] == 1:
            gen_ids = torch.unique(gen_ids[0]).unsqueeze(0)
        score = torch.gather(scores, 1, gen_ids)
        score = torch.where(score < 0, score * self.penalty, score / self.penalty)
        return scores.scatter(1, gen_ids, score)

PREPARED_ROOT = runtime_path("prepared_clinical_dataset", "latest_v4_5000.json")
pointer = json.loads((PREPARED_ROOT / "latest_v4_5000.json").read_text(encoding="utf-8"))
DATASET_VERSION = pointer["version"]
prepared_dir = PREPARED_ROOT / DATASET_VERSION
manifest = json.loads((prepared_dir / "manifest.json").read_text(encoding="utf-8"))
prepared = load_from_disk(str(prepared_dir / "dataset"))
for split, sha in manifest["split_sha256"].items():
    rows_chk = [json.loads(l) for l in (prepared_dir / f"{split}.jsonl").read_text(encoding="utf-8").split("\n") if l.strip()]
    assert digest(rows_chk) == sha, f"Prepared {split} changed"

cfg = manifest["config"]
MODEL_ID = cfg["model"]
MODEL_REVISION = cfg["model_revision"]
SEED = cfg["seed"]
MAX_NEW_TOKENS = 2048
INFERENCE_DTYPE = "float32"
DECODING_CONDITIONS = [
    {"tag": "gen_rp_1.08", "repetition_penalty": 1.08, "generated_only": True},
]

OUTPUT_ROOT = (Path("/content/clinical_v4_5000/results_v4_5000") if Path("/content").is_dir() else Path("results_v4_5000")) / "baseline"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

cases = prepared["validation"].to_list()
assert importlib.metadata.version("transformers") == "4.57.6"

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float32
contract = {
    "dataset_version": DATASET_VERSION,
    "schema_version": cfg["schema_version"],
    "model": MODEL_ID,
    "base_model_revision": MODEL_REVISION,
    "train_sha256": manifest["split_sha256"]["train"],
    "validation_sha256": manifest["split_sha256"]["validation"],
    "case_ids": [c["case_id"] for c in cases],
    "prompt_sha256": digest([c["prompt_text"] for c in cases]),
    "max_new_tokens": MAX_NEW_TOKENS,
    "do_sample": False,
    "dtype": INFERENCE_DTYPE,
    "seed": SEED,
    "tokenizer_add_special_tokens": False,
    "decoding_conditions": DECODING_CONDITIONS,
    "metrics_version": 3,
    "transformers": importlib.metadata.version("transformers"),
    "torch": torch.__version__,
}
contract_path = OUTPUT_ROOT / "comparison_contract.json"
if contract_path.exists():
    assert json.loads(contract_path.read_text(encoding="utf-8")) == contract, "Refusing to reuse outputs under a changed contract"
(OUTPUT_ROOT / "comparison_contract.json").write_text(json.dumps(contract, indent=2), encoding="utf-8")
selection = [{"case_id": c["case_id"], "note": c["note"], "reference": c["reference"], "prompt_text": c["prompt_text"]} for c in cases]
(OUTPUT_ROOT / "selection.json").write_text(json.dumps(selection, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Baseline v3 contract saved ({DATASET_VERSION}): {len(cases)} val cases, device={device}, dtype={dtype}", flush=True)

Baseline v3 contract saved (0d520299c998a045): 100 val cases, device=cuda, dtype=torch.float32


## Parsing, schema and fact-agreement metrics
Malformed whole objects receive zero fact credit. These metrics do not establish clinical accuracy.

In [3]:
from scripts.prepare_v3_dataset import schema_valid
TOP_LEVEL_KEYS = (
    "visit motivation", "admission", "patient information", "patient medical history",
    "surgeries", "symptoms", "medical examinations", "diagnosis tests", "treatments", "discharge",
)
MISSING_VALUES = ("", "none", "null", "unknown", "not provided")

def no_duplicate_keys(pairs):
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError("Duplicate JSON key: " + key)
        result[key] = value
    return result

def reject_constant(value):
    raise ValueError("Nonstandard JSON constant: " + value)

def load_object(text, strict=True):
    obj = json.loads(
        text, strict=strict, object_pairs_hook=no_duplicate_keys, parse_constant=reject_constant
    )
    if not isinstance(obj, dict):
        raise ValueError("Expected a JSON object")
    return obj

def strip_fences(text):
    text = text.strip()
    if text.startswith("```") and text.endswith("```"):
        return re.sub(r"^```(?:json)?\s*", "", text, count=1, flags=re.IGNORECASE)[:-3].strip()
    return text

def facts(value, path=""):
    """Multiset of (field path, normalized value); array order is ignored."""
    result = Counter()
    if isinstance(value, dict):
        for key, child in value.items():
            result.update(facts(child, f"{path}.{key}" if path else key))
    elif isinstance(value, list):
        for child in value:
            result.update(facts(child, path))
    elif value is not None:
        normalized = " ".join(str(value).casefold().split())
        if normalized not in MISSING_VALUES:
            result[(path, normalized)] += 1
    return result

def facts_by_path(value, path=""):
    """Mapping field path -> list of normalized non-missing values."""
    result = {}
    for (p, v), cnt in facts(value, path).items():
        result.setdefault(p, []).extend([v] * cnt)
    return result

def key_paths(value, prefix=""):
    result = set()
    if isinstance(value, dict):
        for key, child in value.items():
            path = f"{prefix}.{key}" if prefix else key
            result.add(path)
            result.update(key_paths(child, path))
    elif isinstance(value, list):
        for child in value:
            result.update(key_paths(child, prefix))
    return result

def _token_f1(a_str, b_str):
    ta = Counter(re.findall(r"[a-z0-9]+", a_str.casefold()))
    tb = Counter(re.findall(r"[a-z0-9]+", b_str.casefold()))
    na, nb = sum(ta.values()), sum(tb.values())
    if not na or not nb:
        return 0.0
    ov = sum((ta & tb).values())
    return (2.0 * ov) / (na + nb)

def soft_fact_scores(expected_obj, actual_obj):
    if actual_obj is None:
        return 0.0, 0.0, 0.0
    exp_map = facts_by_path(expected_obj)
    act_map = facts_by_path(actual_obj)
    n_exp = sum(len(v) for v in exp_map.values())
    n_act = sum(len(v) for v in act_map.values())
    if not n_exp or not n_act:
        return 0.0, 0.0, 0.0
    total_sim = 0.0
    for path, exp_vals in exp_map.items():
        act_vals = act_map.get(path, [])
        if not act_vals:
            continue
        pairs = []
        for i, ev in enumerate(exp_vals):
            for j, av in enumerate(act_vals):
                s = 1.0 if ev == av else _token_f1(ev, av)
                if s > 0:
                    pairs.append((s, i, j))
        pairs.sort(reverse=True)
        used_e, used_a = set(), set()
        for s, i, j in pairs:
            if i not in used_e and j not in used_a:
                used_e.add(i)
                used_a.add(j)
                total_sim += s
    prec = total_sim / n_act
    rec = total_sim / n_exp
    f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) else 0.0
    return prec, rec, f1

def count_duplicate_array_items(text):
    try:
        obj = load_object(strip_fences(text))
    except ValueError:
        return None
    dups = 0
    def walk(node):
        nonlocal dups
        if isinstance(node, list):
            seen = set()
            for item in node:
                key = json.dumps(item, ensure_ascii=False, sort_keys=True)
                if key in seen:
                    dups += 1
                seen.add(key)
            for item in node:
                walk(item)
        elif isinstance(node, dict):
            for v in node.values():
                walk(v)
    walk(obj)
    return dups

def max_word_5gram_repeat(text):
    words = re.findall(r"\S+", text.lower())
    if len(words) < 5:
        return 0
    counts = Counter(tuple(words[i:i+5]) for i in range(len(words) - 4))
    return max(counts.values()) if counts else 0

def evaluate_output(text, reference):
    expected = load_object(reference)
    raw_valid = True
    try:
        load_object(text)
    except ValueError:
        raw_valid = False
    try:
        actual = load_object(strip_fences(text))
        error = None
    except ValueError as exc:
        actual, error = None, str(exc)
    expected_facts = facts(expected)
    actual_facts = facts(actual)
    matches = sum((expected_facts & actual_facts).values())
    n_actual, n_expected = sum(actual_facts.values()), sum(expected_facts.values())
    precision = matches / n_actual if n_actual else 0.0
    recall = matches / n_expected if n_expected else 0.0
    soft_p, soft_r, soft_f1 = soft_fact_scores(expected, actual)
    return {
        "empty": not text.strip(),
        "raw_json_object_valid": raw_valid,
        "json_object_valid_after_fence_removal": actual is not None,
        "parse_error": error,
        "allowed_schema_valid": schema_valid(actual),
        "raw_allowed_schema_valid": raw_valid and schema_valid(actual),
        "exact_top_level_keys": actual is not None and set(actual.keys()) == set(TOP_LEVEL_KEYS),
        "exact_key_match": actual is not None and key_paths(actual) == key_paths(expected),
        "nonmissing_generated_values": n_actual,
        "reference_values": n_expected,
        "matched_values": matches,
        "placeholder_only": actual is not None and n_actual == 0,
        "value_precision": precision,
        "value_recall": recall,
        "value_f1": 2 * precision * recall / (precision + recall) if precision + recall else 0.0,
        "soft_value_precision": soft_p,
        "soft_value_recall": soft_r,
        "soft_value_f1": soft_f1,
        "duplicate_array_items": count_duplicate_array_items(text),
        "max_5gram_repeat": max_word_5gram_repeat(text),
    }

def score_run(cases, outputs, diagnostics):
    records = []
    for case in cases:
        key = case["case_id"]
        text = outputs[key]
        records.append({
            "case_id": key,
            **evaluate_output(text, case["reference"]),
            "reached_budget": bool(diagnostics[key].get("reached_budget")),
            "generated_tokens": int(diagnostics[key]["generated_tokens"]),
            "generated": text,
            "reference": case["reference"],
        })
    frame = pd.DataFrame(records)
    summary = {"selected_cases": len(cases), "evaluated_cases": len(records)}
    for name in (
        "empty", "raw_json_object_valid", "json_object_valid_after_fence_removal",
        "exact_top_level_keys", "allowed_schema_valid", "raw_allowed_schema_valid", "exact_key_match", "placeholder_only", "reached_budget",
    ):
        summary[name + "_cases"] = int(frame[name].sum())
    for name in (
        "value_precision", "value_recall", "value_f1",
        "soft_value_precision", "soft_value_recall", "soft_value_f1",
    ):
        summary["mean_" + name] = float(frame[name].mean())
    for key, col in (
        ("matched_values", "matched_values"),
        ("reference_values", "reference_values"),
        ("generated_values", "nonmissing_generated_values"),
    ):
        summary[key] = int(frame[col].sum())
    summary["mean_generated_tokens"] = float(frame["generated_tokens"].mean())
    summary["max_5gram_repeat_ge_3_cases"] = int((frame["max_5gram_repeat"] >= 3).sum())
    return frame, summary

# Metric self-checks
ref = '{"age":"29","symptoms":[{"name":"chest pain"}]}'
assert evaluate_output(ref, ref)["value_f1"] == 1.0
assert evaluate_output("```json\n" + ref + "\n```", ref)["raw_json_object_valid"] is False
assert evaluate_output("```json\n" + ref + "\n```", ref)["json_object_valid_after_fence_removal"] is True
assert evaluate_output('{"age":"None"}', ref)["placeholder_only"] is True
assert 0.0 < evaluate_output('{"age":"29","symptoms":[{"name":"severe chest pain"}]}', ref)["soft_value_f1"] < 1.0
print("Metric self-checks passed.")

Metric self-checks passed.


## Generate baseline outputs
Save predictions and diagnostics after each case.

In [4]:
token = read_token()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION, token=token)
pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id
random.seed(SEED)
torch.manual_seed(SEED)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=MODEL_REVISION, token=token, dtype=dtype).to(device).eval()
eos = model.generation_config.eos_token_id
eos_ids = eos if isinstance(eos, list) else [eos]

for cond in DECODING_CONDITIONS:
    cond_dir = OUTPUT_ROOT / cond["tag"]
    cond_dir.mkdir(parents=True, exist_ok=True)
    gen_path = cond_dir / "generations.json"
    diag_path = cond_dir / "generation_diagnostics.json"
    generations = json.loads(gen_path.read_text(encoding="utf-8")) if gen_path.exists() else {}
    diagnostics = json.loads(diag_path.read_text(encoding="utf-8")) if diag_path.exists() else {}
    started = time.monotonic()
    for case in cases:
        key = case["case_id"]
        if generations.get(key, "").strip() and key in diagnostics:
            continue
        inputs = tokenizer(case["prompt_text"], return_tensors="pt", add_special_tokens=False).to(device)
        n_prompt = inputs.input_ids.shape[1]
        assert n_prompt + MAX_NEW_TOKENS <= model.config.max_position_embeddings
        with torch.inference_mode():
            logits = model(**inputs, logits_to_keep=1).logits[:, -1, :]
            finite = bool(torch.isfinite(logits).all().item())
            del logits
            if not finite:
                ids = []
            else:
                gen_kwargs = {"max_new_tokens": MAX_NEW_TOKENS, "do_sample": False, "pad_token_id": pad_id}
                if cond["generated_only"] and cond["repetition_penalty"] != 1.0:
                    gen_kwargs["logits_processor"] = LogitsProcessorList([
                        GeneratedOnlyRepetitionPenalty(cond["repetition_penalty"], n_prompt)
                    ])
                generated = model.generate(**inputs, **gen_kwargs)
                ids = generated[0, n_prompt:].tolist()
        text = tokenizer.decode(ids, skip_special_tokens=True)
        diagnostics[key] = {
            "prompt_tokens": n_prompt,
            "generated_tokens": len(ids),
            "finite_first_token_logits": finite,
            "empty_output": not text.strip(),
            "reached_budget": len(ids) >= MAX_NEW_TOKENS,
            "ended_with_eos": bool(ids) and ids[-1] in eos_ids,
        }
        diag_path.write_text(json.dumps(diagnostics, ensure_ascii=False, indent=2), encoding="utf-8")
        if not finite or not text.strip():
            raise RuntimeError(f"base/{cond['tag']}/{key}: failed generation")
        generations[key] = text
        gen_path.write_text(json.dumps(generations, ensure_ascii=False, indent=2), encoding="utf-8")
        print(f"[base | {cond['tag']}] {len(generations)}/{len(cases)} {key}: {len(ids)} toks ({time.monotonic()-started:.1f}s)", flush=True)

print(f"Baseline predictions saved for {len(cases)} cases; cached cases are reused under the verified contract.", flush=True)
del model
if device == "cuda":
    torch.cuda.empty_cache()

tokenizer_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/536M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

Baseline predictions saved for 100 cases; cached cases are reused under the verified contract.


## Save baseline scores and archive

In [5]:
summary_rows = []
for cond in DECODING_CONDITIONS:
    cond_dir = OUTPUT_ROOT / cond["tag"]
    gens = json.loads((cond_dir / "generations.json").read_text(encoding="utf-8"))
    diags = json.loads((cond_dir / "generation_diagnostics.json").read_text(encoding="utf-8"))
    frame, summary = score_run(cases, gens, diags)
    frame.to_csv(cond_dir / "per_case.csv", index=False)
    (cond_dir / "aggregate.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    summary_rows.append({"model": "base", "decoding": cond["tag"], **summary})

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_ROOT / "summary.csv", index=False)
(OUTPUT_ROOT / "summary.json").write_text(json.dumps(summary_rows, indent=2), encoding="utf-8")
archive = shutil.make_archive(str(OUTPUT_ROOT.resolve()), "zip", root_dir=OUTPUT_ROOT.parent, base_dir=OUTPUT_ROOT.name)
display(summary_df)
print("Baseline archive saved:", archive, flush=True)

,model,decoding,selected_cases,evaluated_cases,empty_cases,raw_json_object_valid_cases,json_object_valid_after_fence_removal_cases,exact_top_level_keys_cases,allowed_schema_valid_cases,raw_allowed_schema_valid_cases,...,mean_value_recall,mean_value_f1,mean_soft_value_precision,mean_soft_value_recall,mean_soft_value_f1,matched_values,reference_values,generated_values,mean_generated_tokens,max_5gram_repeat_ge_3_cases
0,base,gen_rp_1.08,100,100,0,0,74,0,0,0,...,0.015247,0.014517,0.044298,0.044121,0.040952,62,3856,2951,882.04,87


Baseline archive saved: /content/clinical_v4_5000/results_v4_5000/baseline.zip
